```{contents}
```
## Weight Initialization 

Weight initialization determines the **starting point** of optimization.
A poor initialization can cause:

* **Vanishing gradients**
* **Exploding gradients**
* Extremely slow convergence
* Model instability

A good initialization preserves the **variance of activations and gradients** across layers so that learning remains stable and efficient.

---

### Mathematical Intuition

For a layer:

$$
z = Wx
$$

We want:

$$
Var(z_l) \approx Var(z_{l-1})
\quad \text{and} \quad
Var(\nabla_l) \approx Var(\nabla_{l+1})
$$

If weights are too small → activations shrink
If weights are too large → activations explode

Initialization controls this balance.

---

### Main Weight Initialization Techniques

| Method          | Best For         | Weight Distribution  | Key Property              |
| --------------- | ---------------- | -------------------- | ------------------------- |
| Zero            | None             | All zeros            | Breaks learning           |
| Random Normal   | Shallow nets     | 𝒩(0, σ²)            | Often unstable            |
| Xavier (Glorot) | Tanh / Sigmoid   | 𝒩(0, 1/fan_in)      | Preserves variance        |
| He (Kaiming)    | ReLU / variants  | 𝒩(0, 2/fan_in)      | Compensates ReLU sparsity |
| Orthogonal      | RNNs / deep nets | Orthonormal          | Preserves gradient norm   |
| LSUV            | Very deep nets   | Layer-wise rescaling | Auto variance fixing      |

---

### PyTorch Demonstrations

#### Zero Initialization (Bad Practice)

```python
def init_zero(m):
    if isinstance(m, nn.Linear):
        nn.init.zeros_(m.weight)
        nn.init.zeros_(m.bias)
```

Leads to identical gradients → no learning.

---

#### Xavier Initialization

```python
def init_xavier(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_normal_(m.weight)
        nn.init.zeros_(m.bias)
```

Used for Sigmoid / Tanh networks.

---

#### He (Kaiming) Initialization

```python
def init_he(m):
    if isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
        nn.init.zeros_(m.bias)
```

Best for ReLU-based architectures.

---

#### Orthogonal Initialization

```python
def init_ortho(m):
    if isinstance(m, nn.Linear):
        nn.init.orthogonal_(m.weight)
```

Helps stabilize very deep networks and RNNs.

---

#### Applying Initialization

```python
model = MyModel()
model.apply(init_he)
```

---

### Experimental Comparison: Gradient Stability

```python
x = torch.randn(64, 100)
model = DeepModel()
model.apply(init_he)

loss = model(x).sum()
loss.backward()

for name, p in model.named_parameters():
    print(name, p.grad.norm())
```

Healthy initialization keeps gradient norms similar across layers.

---

### Initialization vs Activation Compatibility

| Activation  | Recommended Initialization |
| ----------- | -------------------------- |
| Sigmoid     | Xavier                     |
| Tanh        | Xavier                     |
| ReLU        | He                         |
| LeakyReLU   | He                         |
| SELU        | LeCun normal               |
| Transformer | Xavier + LayerNorm         |

---

### Advanced Variants

| Variant               | Purpose                             |
| --------------------- | ----------------------------------- |
| LSUV                  | Automatic variance normalization    |
| LeCun Normal          | Self-normalizing networks           |
| Scaled Initialization | Transformers                        |
| Fixup Init            | Residual nets without normalization |

---

### Practical Guidelines

* Match initialization to activation function.
* Use He initialization for almost all modern ReLU models.
* Avoid zero or naive random initialization.
* Monitor gradient norms early in training.

---

### Key Takeaway

> **Good weight initialization is the foundation of stable deep learning.**
> It ensures effective signal propagation, faster convergence, and reliable optimization.
